# 04 - SQL Multi-Table Analysis

Integrating three supplementary tables (Drivers, Hubs, Vehicles) with the main orders dataset using SQLite, and testing whether driver- and vehicle-level attributes actually predict delivery performance. Built using SQL joins and aggregations rather than pandas, as a deliberate exercise in SQL fundamentals.


## Setup

In [2]:
import pandas as pd
import numpy as np
import sqlite3

df = pd.read_csv('/Users/DianaLara/PycharmProjects/PythonLearning_DataAnalysis/Orders.csv', encoding='UTF-16', sep='\t')

df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
df['Actual Delivery Date'] = pd.to_datetime(df['Actual Delivery Date'], dayfirst=True, errors='coerce')

df.head()


,Order ID,Actual Delivery Date,Delay Reason,Driver ID,Driver Name,Hub Name,Is Delayed,Is On Time,Order Date,Order Status,Vehicle Name,Vehicle Type,Customer Satisfaction Score,Delivery Time Hours,Hub Processing Time Hours
0,1,2024-10-25,NaN,43,Karen Rodriguez,San Antonio Hub,False,True,2024-10-25,Delivered,FT-036,Truck,4,6.81,0.89
1,2,2024-06-16,NaN,29,Matthew Williams,Houston Hub,False,True,2024-06-16,Delivered,FT-016,Van,4,5.74,3.60
2,3,2024-07-05,NaN,25,Nancy Harris,Austin Hub,False,True,2024-07-05,Delivered,FT-040,Van,4,12.91,2.07
3,4,2023-08-22,NaN,20,David Davis,Fort Worth Hub,False,True,2023-08-22,Delivered,FT-039,Van,5,9.40,2.37
4,5,2024-06-06,Severe Weather,49,Joseph Williams,Dallas Main Hub,True,False,2024-06-02,Delivered,FT-018,Truck,3,103.48,1.80


## 1. Loading Supplementary Tables

Three additional CSVs provide driver, hub, and vehicle detail not present in the main orders table. Each required different loading fixes:
- `Drivers.csv` and `Hubs.csv` were UTF-16 encoded and tab-separated, despite the `.csv` extension - likely exported from a Windows tool.
- `Vehicles.csv` loaded cleanly with the standard defaults.


In [3]:
drivers_df = pd.read_csv('/Users/DianaLara/PycharmProjects/PythonLearning_DataAnalysis/Drivers.csv', encoding='utf-16', sep='\t')
print(drivers_df.columns)


Index(['DriverID', 'DriverName', 'Employment Type', 'Hire Date',
       'Experience Years', 'Performance Rating'],
      dtype='str')


In [4]:
hubs_df = pd.read_csv('/Users/DianaLara/PycharmProjects/PythonLearning_DataAnalysis/Hubs.csv', encoding='utf-16', sep='\t')
print(hubs_df.columns)


Index(['Hub ID', 'HubName', 'Hub Capacity'], dtype='str')


In [6]:
vehicles_df = pd.read_csv('/Users/DianaLara/PycharmProjects/PythonLearning_DataAnalysis/Vehicles.csv', encoding='utf-16', sep='\t')
print(vehicles_df.columns)


Index(['Purchase Date', 'Vehicle ID', 'Vehicle Model', 'Vehicle Status',
       'Breakdown', 'Maintenance count Alert', 'Vehicle Code'],
      dtype='str')


### 1.1 Identifying join keys

Column names differ between tables (e.g. `Driver ID` vs. `DriverID`), so join keys were confirmed by comparing actual values, not just column names. `Vehicle Code` was confirmed as the correct match for `Vehicle Name` (both use the `FT-0XX` format) rather than `Vehicle ID`, which is a separate internal ID.

| Table | Join column | Matches main `df` column |
|---|---|---|
| `drivers_df` | `DriverID` | `Driver ID` |
| `hubs_df` | `HubName` | `Hub Name` |
| `vehicles_df` | `Vehicle Code` | `Vehicle Name` |


## 2. Setting Up SQLite

Rather than merging tables in pandas, all four tables are loaded into a SQLite database so the remaining analysis can be done in real SQL: joins, aggregations, and filtering.


In [7]:
conn = sqlite3.connect('delivery_analysis.db')

df.to_sql('orders', conn, if_exists='replace', index=False)
drivers_df.to_sql('drivers', conn, if_exists='replace', index=False)
hubs_df.to_sql('hubs', conn, if_exists='replace', index=False)
vehicles_df.to_sql('vehicles', conn, if_exists='replace', index=False)

print("Tables loaded successfully")


Tables loaded successfully


In [ ]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)


## 3. Does Driver Performance Rating Predict Outcomes?

Testing whether the `Performance Rating` field in `Drivers.csv` (a company-assigned rating) actually corresponds to real delivery outcomes.


In [8]:
query = """
SELECT d.[Performance Rating],
       COUNT(*) AS total_orders,
       ROUND(AVG(o.[Delivery Time Hours]), 2) AS avg_delivery_hours,
       ROUND(AVG(o.[Customer Satisfaction Score]), 2) AS avg_satisfaction
FROM orders o
JOIN drivers d ON o.[Driver ID] = d.DriverID
GROUP BY d.[Performance Rating]
ORDER BY d.[Performance Rating] DESC
"""
result = pd.read_sql(query, conn)
print(result)


   Performance Rating  total_orders  avg_delivery_hours  avg_satisfaction
0                   5          5147               35.70              4.17
1                   4         11335               35.86              4.17
2                   3          9397               35.81              4.17
3                   2          1269               35.65              4.20
4                   1           831               35.03              4.18


**Finding:** `Performance Rating` shows essentially no relationship with actual delivery outcomes. Avg delivery time barely varies across all five rating levels (35.03-35.86 hrs), and satisfaction is flat (4.17-4.20) regardless of rating. If anything, the lowest-rated drivers (rating 1) have a slightly *faster* average delivery time than higher-rated drivers, the opposite of what the rating would suggest. This indicates the rating likely reflects criteria not captured in this dataset (e.g. subjective review, tenure, soft skills) rather than measurable delivery performance.


## 4. Does Driver Experience Predict Outcomes?

A more objective driver attribute than a subjective rating - testing whether it tells a different story.


In [9]:
query = """
SELECT d.[Experience Years],
       COUNT(*) AS total_orders,
       ROUND(AVG(o.[Delivery Time Hours]), 2) AS avg_delivery_hours,
       ROUND(AVG(o.[Customer Satisfaction Score]), 2) AS avg_satisfaction
FROM orders o
JOIN drivers d ON o.[Driver ID] = d.DriverID
GROUP BY d.[Experience Years]
ORDER BY d.[Experience Years] DESC
"""
result = pd.read_sql(query, conn)
print(result)


   Experience Years  total_orders  avg_delivery_hours  avg_satisfaction
0                10          1479               36.46              4.17
1                 7          2878               36.29              4.15
2                 6          2455               34.85              4.18
3                 5          4041               36.00              4.15
4                 4          5357               35.62              4.18
5                 3          5294               36.06              4.15
6                 2          4756               35.57              4.19
7                 1          1719               35.39              4.18


**Finding:** same null result. Delivery time bounces narrowly between 34.85 and 36.46 hours with no trend as experience increases, and satisfaction stays flat (4.15-4.19) regardless of tenure. Two different driver-quality metrics (rating and experience) both show no relationship with outcomes. This is a consistent pattern rather than a single fluke, which strengthens the conclusion that operational results are driven by factors outside these driver attributes.


## 5. Does Vehicle Status Predict Outcomes?

Shifting focus to vehicles, since `Vehicle Breakdown` was identified as a top delay reason in notebook 01.


In [10]:
query = """
SELECT v.[Vehicle Status],
       COUNT(*) AS total_orders,
       ROUND(AVG(o.[Delivery Time Hours]), 2) AS avg_delivery_hours,
       ROUND(AVG(o.[Customer Satisfaction Score]), 2) AS avg_satisfaction
FROM orders o
JOIN vehicles v ON o.[Vehicle Name] = v.[Vehicle Code]
GROUP BY v.[Vehicle Status]
ORDER BY avg_delivery_hours DESC
"""
result = pd.read_sql(query, conn)
print(result)


  Vehicle Status  total_orders  avg_delivery_hours  avg_satisfaction
0    Maintenance          7581               35.93              4.16
1         Active         20398               35.72              4.17


**Finding:** a very small gap in the expected direction (vehicles currently in "Maintenance" status: 35.93 avg hours / 4.16 satisfaction vs. "Active": 35.72 / 4.17), but the difference is minor. **Caveat:** `Vehicle Status` may reflect the vehicle's status at the time the data was pulled, not its status at the time of each specific historical delivery; this timing mismatch is a limitation worth noting.


In [11]:
# Check whether 'Maintenance count Alert' is a true cumulative count or a simple flag
print(vehicles_df['Maintenance count Alert'].unique())


[0 1]


**Finding:** `Maintenance count Alert` only contains values `[0, 1]` which indicates that despite its name implying a cumulative count, it's actually just a binary flag. Grouping by it produced results identical to grouping by `Vehicle Status` (7,581 / 20,398 order split in both cases), confirming it is redundant with that column rather than a distinct signal. A reminder that column names don't always accurately describe their contents. Worth verifying rather than assuming.


## 6. Does Vehicle Status Predict Breakdown-Specific Delays?

The most direct test: does the delay reason 'Vehicle Breakdown' actually cluster on vehicles currently flagged as needing maintenance?


In [12]:
query = """
SELECT v.[Vehicle Status], o.[Delay Reason], COUNT(*) AS count
FROM orders o
JOIN vehicles v ON o.[Vehicle Name] = v.[Vehicle Code]
WHERE o.[Delay Reason] = 'Vehicle Breakdown'
GROUP BY v.[Vehicle Status]
"""
result = pd.read_sql(query, conn)
print(result)


  Vehicle Status       Delay Reason  count
0         Active  Vehicle Breakdown    461
1    Maintenance  Vehicle Breakdown    162


**Finding:** normalizing for order volume, breakdown-delay rates are nearly identical between the two groups:
- Active vehicles: 461 / 20,398 orders = **2.26%**
- Maintenance vehicles: 162 / 7,581 orders = **2.14%**

If anything, vehicles currently flagged "Maintenance" show a very slightly *lower* breakdown-delay rate, the opposite of the intuitive expectation. This suggests breakdown-caused delays may be closer to random, unpredictable events than something reliably indicated by a vehicle's current status.


## Summary of Key Findings

**Data integration**
- Successfully joined four tables (orders, drivers, hubs, vehicles) via SQLite, using confirmed join keys (`DriverID`, `HubName`, `Vehicle Code`) rather than assuming column names implied a match.
- `Drivers.csv` and `Hubs.csv` required UTF-16 encoding and tab-separator fixes despite the `.csv` extension.

**Core finding - a consistent null result across four "quality" signals**
- `Performance Rating`, `Experience Years`, `Vehicle Status`, and `Maintenance count Alert` (redundant with Vehicle Status) all show little to no relationship with actual delivery time, satisfaction, or breakdown-delay rate.
- This is a stronger conclusion than any single null result alone, since the same pattern held across multiple, independent attributes on both the driver and vehicle side.
- **Implication:** operational outcomes in this dataset appear to be driven by factors other than the driver/vehicle attributes captured here. Possibly route, hub, timing, or external conditions (weather, traffic) already captured in `Delay Reason`, or simply normal variation.

**Limitations noted**
- `Vehicle Status` may represent a current snapshot rather than status at time of delivery, which could understate any real relationship, flagged as a limitation rather than treated as disproof.
- Column naming isn't always a reliable guide to what a field actually measures (`Maintenance count Alert`).

**Next steps:** explore `Hubs.csv`'s `Hub Capacity` field against order volume per hub, to test whether hubs operating over/under capacity show different performance; consider whether `Order Date` timing could be used to approximate vehicle status at time of delivery rather than relying on the current snapshot.
